# Analyses Croisées Météo, Mobilité et Pollution - UrbanHub (Rôle 3)

Ce notebook présente les analyses corrélatives entre la météorologie (NOAA), la tension d'usage des vélos (CityBikes) et les concentrations de pollution atmosphérique (OpenAQ) pour les villes françaises de notre périmètre.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import glob
import os

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

## 1. Chargement de la table Gold Croisée

In [2]:
gold_dir = os.path.join("data", "urbanhub-gold", "cross")
files = glob.glob(os.path.join(gold_dir, "city=*", "merged_daily.parquet"))

dfs = []
for f in files:
    df = pd.read_parquet(f)
    dfs.append(df)

if dfs:
    df_cross = pd.concat(dfs, ignore_index=True)
    print(f"Chargement réussi : {len(df_cross)} lignes de données croisées.")
    print(df_cross.head())
else:
    print("Aucun fichier Gold croisé trouvé. Exécutez merge_datasets.py au préalable !")

Aucun fichier Gold croisé trouvé. Exécutez merge_datasets.py au préalable !


## 2. Analyse des Corrélations (Météo, Vélos & Pollution)

In [3]:
if dfs:
    # Sélection des variables numériques clés
    cols = ['avg_temperature', 'avg_wind_speed', 'avg_pressure', 'avg_visibility', 
            'avg_pm25', 'avg_pm10', 'avg_no2', 'avg_usage_pressure', 'critical_stations_count']
    
    corr = df_cross[cols].corr()
    
    # Heatmap de corrélation
    sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f', vmin=-1, vmax=1)
    plt.title("Matrice de corrélation croisée (Météo / Mobilité / Pollution)")
    plt.tight_layout()
    plt.show()

## 3. Impact de la Météo sur la Pollution

Nous analysons comment la vitesse du vent influence la dispersion des particules fines (PM2.5).

In [4]:
if dfs:
    sns.scatterplot(data=df_cross, x='avg_wind_speed', y='avg_pm25', hue='city', alpha=0.8)
    plt.title("Dispersion des PM2.5 en fonction de la vitesse du vent")
    plt.xlabel("Vitesse du vent (m/s / km/h)")
    plt.ylabel("PM2.5 (µg/m³)")
    plt.tight_layout()
    plt.show()

## 4. Impact de la Température sur l'usage des vélos

In [5]:
if dfs:
    sns.lmplot(data=df_cross, x='avg_temperature', y='avg_usage_pressure', hue='city', height=6, aspect=1.5)
    plt.title("Tension d'usage vélo en fonction de la température extérieure")
    plt.xlabel("Température moyenne (°C)")
    plt.ylabel("Tension d'usage vélo (1 = saturé / 0 = libre)")
    plt.tight_layout()
    plt.show()